# AI Tools for Actuaries
## Exercise of lecture 9: LocalGLMnet
### Author: Marco Maggi, Mario Wuthrich
### Version Summer School August/September 2026

In [ ]:
# Import required libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import glm

pd.options.mode.chained_assignment = None

# Set random seed
rng = np.random.default_rng(500)

### Load data

In [ ]:
# Load the data (load the data with the entity embeddings)
df = pd.read_parquet("../Data/freMTPL2freqEmb.parquet")
df.head()

In [ ]:
# Add random component
df["RandN"] = rng.normal(0, 1, size=len(df))

In [ ]:
learn = df[df["LearnTest"] == "L"]
test = df[df["LearnTest"] == "T"]

### Pre-process data for LocalGLMnet

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OrdinalEncoder,
    StandardScaler,
)
from sklearn.decomposition import PCA


def clip_and_scale(upper):
    pipe = Pipeline(
        steps=[
            ("clip", FunctionTransformer(lambda x: np.clip(x, a_min=0, a_max=upper))),
            ("scale", StandardScaler()),
        ]
    )
    return pipe


def make_pca_pipe(names):
    return Pipeline(
        steps=[
            ("pca", PCA(n_components=len(names))),
            (
                "rename",
                FunctionTransformer(
                    # If input is DataFrame (from set_output), use set_axis. Else create DataFrame.
                    lambda x: x.set_axis(names, axis=1)
                    if hasattr(x, "set_axis")
                    else pd.DataFrame(x, columns=names),
                    check_inverse=False,
                ),
            ),
            ("scale", StandardScaler()),
        ]
    )


density = Pipeline(
    steps=[
        ("log", FunctionTransformer(lambda x: np.log(x).round(2))),
        ("scale", StandardScaler()),
    ]
)

area = Pipeline(
    steps=[
        ("encode", OrdinalEncoder()),
        ("scale", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "clip_and_scale",
            clip_and_scale([20, 90, 150, 15]),
            ["VehAge", "DrivAge", "BonusMalus", "VehPower"],
        ),
        (
            "pca_veh_brand",
            make_pca_pipe(["VehBrandEmb1", "VehBrandEmb2"]),
            ["VehBrandEmb1", "VehBrandEmb2"],
        ),
        (
            "pca_region",
            make_pca_pipe(["RegionEmb1", "RegionEmb2"]),
            ["RegionEmb1", "RegionEmb2"],
        ),
        ("area", area, ["Area"]),
        ("density", density, ["Density"]),
        (
            "veh_gas",
            FunctionTransformer(lambda x: (x == "Diesel").astype(np.float32)),
            ["VehGas"],
        ),
        ("veh_brand", OrdinalEncoder(), ["VehBrand", "Region"]),
        ("passthrough", "passthrough", ["RandN", "ClaimNb"]),
    ],
    verbose_feature_names_out=False,
)


# Just a check: Fit preprocessor to training data and apply to some lines from test
preprocessor.set_output(transform="pandas").fit(learn)
preprocessor.transform(test.head())


In [ ]:
print(f"Learning set size: {len(learn)}")
print(f"Test set size: {len(test)}")

In [ ]:
X_learn = preprocessor.fit_transform(learn)
X_learn.head(3)

### GLM baseline analysis

In [ ]:
# We start with a baseline GLM to initialize the LocalGLMnet suitably

# Features
features = [
    "Area",
    "VehPower",
    "VehAge",
    "DrivAge",
    "BonusMalus",
    "VehGas",
    "Density",
    "VehBrandEmb1",
    "VehBrandEmb2",
    "RegionEmb1",
    "RegionEmb2",
    "RandN",
]

# Fit a Poisson GLM using the package statsmodels.
# Set `ClaimNb` as response variable and `features` as covariates.
# Use log `Exposure` as offset.
model = ...  # replace ... with your code

glm_results = model.fit()

# Display model summary
glm_results.summary()

### Calculate deviance losses

In [ ]:
X_test = preprocessor.transform(test)

In [ ]:
from sklearn.metrics import mean_poisson_deviance

# Get predictions
learn["GLM"] = glm_results.predict(X_learn)
test["GLM"] = glm_results.predict(X_test)

# Calculate in-sample and out-of-sample deviance
learn_deviance = 100 * mean_poisson_deviance(
    learn["ClaimNb"] / learn["Exposure"], learn["GLM"], sample_weight=learn["Exposure"]
)
test_deviance = 100 * mean_poisson_deviance(
    test["ClaimNb"] / test["Exposure"], test["GLM"], sample_weight=test["Exposure"]
)

print("Deviance Losses:")
print(f"Learning sample: {learn_deviance:.3f}")
print(f"Test sample: {test_deviance:.3f}")

## LocalGLMnet
We have performed all the preparatory work above, and now we dive into the LocalGLMnet

### Define the LocalGLMnet arichtecture (of depth 3)

In [ ]:
import torch
from torch import nn
from torch.nn import init

simpel = nn.Linear(2, 4)
simpel.weight

In [ ]:
import torch
from torch import nn
from torch.nn import init


class FNN(nn.Module):
    def __init__(self, seed, n_features, hidden_layers, intercept, glm_coefs):
        super().__init__()
        torch.manual_seed(seed)
        self.hidden_layers = nn.ModuleList()
        for i in range(len(hidden_layers)):
            if i == 0:
                self.hidden_layers.append(nn.Linear(n_features, hidden_layers[i]))
            else:
                self.hidden_layers.append(
                    nn.Linear(hidden_layers[i - 1], hidden_layers[i])
                )
        # Define the last layer of the neural network which fits the
        # `attention weights`
        self.local_glm = ...  # replace ... with your code
        # Initialize the weights and biases of the last layer such that the SGD
        # will start from the MLE estimates of the GLM.
        self.local_glm.bias.data = ...  # replace ... with your code
        init.constant_(..., 0.0)  # replace ... with your code
        # Define the intercept as trainable parameter and initialize it with the GLM intercept.
        self.intercept = nn.Parameter(...)  # replace ... with your code

    def forward(self, design, v, get_attentions=False):
        # Implement the forward pass of the LocalGLMnet.
        x = torch.tanh(self.hidden_layers[0](design))
        for layer in self.hidden_layers[1:]:
            x = ...  # replace ... with your code
        x = ...  # replace ... with your code
        if get_attentions:
            pass  # replace pass with your code
        skip_connection = torch.einsum("ij,ij->i", x, design).unsqueeze(1)
        x = ...  # replace ... with your code
        return torch.exp(x).flatten() * v


SEED = 21456783
M_FEAT = len(features)  # number of features
HIDDEN = [20, 15, 10]
# Create model with three hidden layers
model = FNN(
    SEED,
    n_features=M_FEAT,
    hidden_layers=HIDDEN,
    intercept=glm_results.params.iloc[0],
    glm_coefs=glm_results.params.to_numpy()[1:],
)


### Check that the LocalGLMnet before training replicates the GLM.

In [ ]:
# Backtest the initalization
model.eval()
X_test_tensor = torch.tensor(X_test[features].values, dtype=torch.float32)
X_learn_tensor = torch.tensor(X_learn[features].values, dtype=torch.float32)
exposure_test_tensor = torch.tensor(
    test["Exposure"].astype("float32").values, dtype=torch.float32
)
exposure_learn_tensor = torch.tensor(
    learn["Exposure"].astype("float32").values, dtype=torch.float32
)

# call the LocalGLMnet to get predictions on test and learn data.
# replace None with your code.
test_GLM = None.detach().numpy()
learn_GLM = None.detach().numpy()

# Exposure
V_learn = learn["Exposure"]
V_test = test["Exposure"]

# Response
Y_learn = learn["ClaimNb"]
Y_test = test["ClaimNb"]

poisson_deviance_train_glm = 100 * mean_poisson_deviance(
    Y_learn / V_learn, learn_GLM / V_learn, sample_weight=V_learn
)
poisson_deviance_test_glm = 100 * mean_poisson_deviance(
    Y_test / V_test, test_GLM / V_test, sample_weight=V_test
)
print(
    "Poisson Deviance (Train, Test):",
    round(poisson_deviance_train_glm, 3),
    round(poisson_deviance_test_glm, 3),
)

### Train the LocalGLMnet model

In [ ]:
from copy import deepcopy


def train_model(
    model,
    X_train,
    y_train,
    v_train,
    X_val,
    y_val,
    v_val,
    optimizer,
    checkpoint_path,
    batch_size,
    n_epochs=100,
):
    loss_fn = nn.PoissonNLLLoss(log_input=False, reduction="sum")
    best_val_loss = float("inf")
    history = {"loss": [], "val_loss": []}

    # Create dataset indices for batching
    num_batches = (len(X_train) + batch_size - 1) // batch_size

    for epoch in range(n_epochs):
        # Training phase
        model.train()
        epoch_loss = 0.0
        # Indices can be shuffled for each epoch. We don't do it here.
        indices = torch.arange(len(X_train))

        for i in range(num_batches):
            # Get batch indices
            batch_indices = indices[
                i * batch_size : min((i + 1) * batch_size, len(X_train))
            ]

            # Get batch data
            X_batch = X_train[batch_indices]
            v_batch = v_train[batch_indices]
            y_batch = y_train[batch_indices]

            # Forward pass
            pred_batch = model(X_batch, v_batch)
            loss = loss_fn(pred_batch, y_batch)

            # Backward pass and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        # Average loss for the epoch
        epoch_loss /= v_train.sum().item()
        history["loss"].append(epoch_loss)

        # Validation phase
        model.eval()
        with torch.no_grad():
            pred_val = model(X_val, v_val)
            val_loss = (loss_fn(pred_val, y_val) / v_val.sum()).item()
            history["val_loss"].append(val_loss)

        # Store best model
        if val_loss < best_val_loss and isinstance(checkpoint_path, str):
            best_val_loss = val_loss
            best_model = deepcopy(model)

        # Print progress
        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch {epoch + 1}/{n_epochs}, Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}"
            )
    torch.save(best_model.state_dict(), checkpoint_path)
    return history


In [ ]:
from sklearn.model_selection import train_test_split

convert_to_tensor = lambda x: torch.tensor(x.values, dtype=torch.float32)
train, val = train_test_split(learn, test_size=0.1, random_state=125548)

X_learn = convert_to_tensor(preprocessor.transform(learn)[features])
X_train = convert_to_tensor(preprocessor.transform(train)[features])
X_val = convert_to_tensor(preprocessor.transform(val)[features])
X_test = convert_to_tensor(preprocessor.transform(test)[features])

y_learn, v_learn = convert_to_tensor(learn.ClaimNb), convert_to_tensor(learn.Exposure)
y_train, v_train = convert_to_tensor(train.ClaimNb), convert_to_tensor(train.Exposure)
y_val, v_val = convert_to_tensor(val.ClaimNb), convert_to_tensor(val.Exposure)
y_test, v_test = convert_to_tensor(test.ClaimNb), convert_to_tensor(test.Exposure)


In [ ]:
optimizer = torch.optim.NAdam(model.parameters())
checkpoint_path = f"./Networks/LocalGLMnet_{SEED}.pt"
history = train_model(
    model,
    X_train,
    y_train,
    v_train,
    X_val,
    y_val,
    v_val,
    optimizer,
    checkpoint_path,
    batch_size=5_000,
    n_epochs=100,
)


In [ ]:
# Plot training history (vertical line at best validation loss)
fig = (
    pd.DataFrame({"loss": history["loss"], "val_loss": history["val_loss"]})
    .rename(columns={"loss": "Training", "val_loss": "Validation"})
    .plot(xlabel="Epoch - 1", ylabel="Loss", title="Loss During Training", grid=True)
)
_ = fig.axvline(np.argmin(history["val_loss"]), color="black", linestyle="--")


### LocalGLMnet results

In [ ]:
from sklearn.metrics import mean_poisson_deviance


# Helper functions to evaluate the model via average Poisson deviance
def score(model, X, y, v):
    """Evaluate the model using sklearn's mean_poisson_deviance."""
    pred = model(X, v).detach().numpy()
    return 100 * mean_poisson_deviance(
        y.detach().numpy() / v, pred / v, sample_weight=v
    )


# Load best weights and evaluate
model.load_state_dict(torch.load(checkpoint_path))
model.eval()

print("===GLM===")
print(
    "Poisson Deviance (Train, Test):",
    round(poisson_deviance_train_glm, 3),
    round(poisson_deviance_test_glm, 3),
)
print("===LocalGLMnet===")
print(f"Poisson Deviance (Learn): {score(model, X_learn, y_learn, v_learn):.3f}")
print(f"Poisson Deviance (Test): {score(model, X_test, y_test, v_test):.3f}")

### Illustrate the LocalGLMnet results: extract attention weights

In [ ]:
# replace None with your code
attention_weights = None.detach().numpy()
attention_df = pd.DataFrame(attention_weights, columns=features)

### Boxplot of the attention weights (RandNX is unrelated to the response)

In [ ]:
# Calculate standard deviation of RandNX (which does not impact the response)
randnx_std = attention_df["RandN"].std()
threshold = 2.576 * randnx_std

# Plot attention weights
plt.figure(figsize=(12, 6))
attention_df.boxplot()
plt.xticks(rotation=45)
plt.title("boxplot of attention weights")
plt.axhline(y=threshold, color="r", linestyle="--", label="+2.576 std dev (99% CI)")
plt.axhline(y=-threshold, color="r", linestyle="--", label="-2.576 std dev (99% CI)")
plt.axhline(y=0, color="k", linestyle="-", label="zero line")
plt.legend()
plt.tight_layout()
plt.show()

### Compute importance measure for all variables/terms

In [ ]:
# Calculate and plot the variable importance measure

### Plot individual attention weights for selected variables

In [ ]:
# Function to create individual attention weight plots
def plot_attention_weights(feature_name, alpha):
    # Create the plot
    plt.figure(figsize=(10, 6))

    # Plot the attention weights
    plt.scatter(
        test[feature_name],
        attention_df[feature_name],
        alpha=0.5,
        s=20,
        label="attention weights",
    )

    # Add reference lines
    plt.axhline(y=0, color="cyan", linestyle="-", label="zero line")
    plt.axhline(
        y=0.674 * randnx_std,
        color="orange",
        linestyle="-",
        label="0.674 std.dev. (50%)",
    )
    plt.axhline(y=-0.674 * randnx_std, color="orange", linestyle="-")
    plt.axhline(
        y=2.576 * randnx_std, color="red", linestyle="-", label="2.576 std.dev. (99%)"
    )
    plt.axhline(y=-2.576 * randnx_std, color="red", linestyle="-")

    # Add shaded area
    plt.fill_between(
        [test[feature_name].min(), test[feature_name].max()],
        [-0.674 * randnx_std, -0.674 * randnx_std],
        [0.674 * randnx_std, 0.674 * randnx_std],
        color="orange",
        alpha=0.3,
    )

    # Add local regression fit
    from statsmodels.nonparametric.smoothers_lowess import lowess

    # Sort the data for local regression
    sorted_indices = np.argsort(test[feature_name])
    x_sorted = test[feature_name].iloc[sorted_indices]
    y_sorted = attention_df[feature_name].iloc[sorted_indices]

    # Fit local regression
    lowess_fit = lowess(y_sorted, x_sorted, frac=alpha, it=3)

    # Plot the local regression fit
    plt.plot(
        lowess_fit[:, 0],
        lowess_fit[:, 1],
        color="lightgreen",
        label="local regression fit",
    )

    # Customize the plot
    plt.title(f"attention weights: {feature_name}", fontsize=14)
    plt.xlabel(feature_name, fontsize=12)
    plt.ylabel("attention weights", fontsize=12)
    plt.legend(loc="lower right")

    # Set y-axis limits
    # ylim0 = max(abs(attention_df[feature_name]))
    ylim0 = np.max(np.abs(attention_df))
    plt.ylim(-ylim0, ylim0)

    plt.tight_layout()
    plt.show()


In [ ]:
# This is the purely random variable not impacting the response
# we perform a local regression which is a bit time consuming
plot_attention_weights("RandN", 0.3)

In [ ]:
# BonusMalus is the most significant term
# the local regression is not fully sensible because BonusMalus values cluster at the lowest level
plot_attention_weights("BonusMalus", 0.6)

In [ ]:
# Driver age variabel
plot_attention_weights("DrivAge", 0.3)

In [ ]:
col1 = "DrivAge"
col2 = "BonusMalus"
y = attention_df[col1]
mask = rng.choice(range(attention_df.shape[0]), size=5000, replace=False)
x = test[col1]
color = test[col2]
color_median = 60
color_binary = (color >= color_median).astype(int)
q1, q2, q3 = 51, 65, 85
color_bins = pd.cut(
    color, bins=[-np.inf, q1, q2, q3, np.inf], labels=[0, 1, 2, 3], include_lowest=True
).astype(int)
fig, ax = plt.subplots(figsize=(8, 6))
# ax.set_title(col)
scatter = ax.scatter(
    x.values[mask],
    y.values[mask],
    s=1,
    c=color_bins.values[mask],
    cmap="RdYlBu",
    alpha=0.8,
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_ticks([0, 1, 2, 3])
cbar.set_ticklabels(
    [
        f"< {q1:.1f}",
        f"{q1:.1f}-{q2:.1f}",
        f"{q2:.1f}-{q3:.1f}",
        f">= {q3:.1f}",
    ]
)
cbar.set_label(f"{col2}")
ax.set_ylabel("$\\beta(\\boldsymbol{x})$")
ax.set_xlabel(col1)

### Gradient of interactions

In [ ]:
import torch.autograd as autograd

n, p = X_test.shape
gradients = np.empty((p, n, p))
input_tensor = X_test

input_tensor.requires_grad = True
attentions = model(X_test, v_test, get_attentions=True)
for i in range(p):
    grad_scaling = torch.ones_like(attentions[:, i])
    gradient_i = autograd.grad(
        attentions[:, i], input_tensor, grad_scaling, create_graph=True
    )
    gradients[i, :, :] = gradient_i[0].numpy(force=True)

In [ ]:
# plot the gradient of the attention for the variable `DrivAge`

## Ensemble

Train the same neural network using different random seeds, which impact the initial weights of the hidden layers.

In [ ]:
seeds = [
    1752305036,
    4284935567,
    909886011,
    4253642063,
    3875387572,
    2984734056,
    56601707,
    803726624,
    215740934,
    1236640324,
]
for seed in seeds:
    model = ...  # replace ... with your code
    optimizer = torch.optim.NAdam(model.parameters())
    checkpoint_path = f"./Networks/LocalGLMnet_{seed}.pt"
    history = ...  # replace ... with your code


Get the derivatives of the attentions with respect to the input features for each trained network, then take the average over all trained networks.

In [ ]:
n, p = X_test.shape
gradients = np.empty((10, p, n, p))
input_tensor = X_test
input_tensor.requires_grad = True
for k, seed in enumerate(seeds):
    checkpoint_path = f"./Networks/LocalGLMnet_{seed}.pt"
    model.load_state_dict(torch.load(checkpoint_path))
    attentions = model(X_test, v_test, get_attentions=True)
    for i in range(p):
        grad_scaling = torch.ones_like(attentions[:, i])
        gradient_i = autograd.grad(
            attentions[:, i], input_tensor, grad_scaling, create_graph=True
        )
        gradients[k, i, :, :] = gradient_i[0].numpy(force=True)

In [ ]:
# plot the gradient of the attention for the variable `DrivAge`, now based on the ensemble model

In [ ]:
# plot the derivative of the attention for the variable `DrivAge` with respect to `BonusMalus`, showing the results of each model of the ensemble